
# Zero-shot Benchmark — **SigLIP-B** (open_clip)

Variante coerente con i notebook CLIP che hai già: stessa pipeline, stesse metriche/visualizzazioni e salvataggi su CSV.
Pensato per **Colab + Google Drive**.

**Cosa fa**
- Carica **SigLIP-B** via `open_clip` (backbone ViT-B/16, 224px)
- Esegue **zero-shot** (prompt ensemble + temperature search)
- Calcola: **Top-1**, **Macro-F1**, **Top-5**, **Balanced Accuracy**
- Stima **Latency (ms/img)**, registra **GPU-h=0.0**, **Effort (h)** e **Note**
- Appende una riga a `benchmark_results.csv` (in `/content` e nella cartella del dataset su Drive)


In [ ]:

# === Install ===
!pip -q install open_clip_torch scikit-learn matplotlib pandas tqdm


In [ ]:

# === Imports & Config ===
import os, time, json, math
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import open_clip
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import (accuracy_score, classification_report, 
                             top_k_accuracy_score, f1_score, balanced_accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

# Colab + Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# --------- CONFIG (MODIFICA QUI) ---------
DATA_ROOT = "/content/drive/MyDrive/ArchiStyles-2"  # cartella dataset
SPLIT = "test"  # "test" | "valid" | "train"
BATCH_SIZE = 64

# Modello SigLIP-B in open_clip
MODEL_NAME = "ViT-B-16-SigLIP"   # variante SigLIP base (224px)
PRETRAINED = "webli"             # checkpoint webli/LAION (in open_clip)

# Prompt templates
TEMPLATES = [
    "a photo of a {} building",
    "a photograph of {} architecture",
    "an example of {} architecture",
    "a building in {} style",
]

# Effort & note per scorecard
EFFORT_HOURS = 0.5
NOTES = "SigLIP-B zero-shot, prompt ensemble + temperature search"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


In [ ]:

# === Model & Dataset ===
# Model
model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=MODEL_NAME,
    pretrained=PRETRAINED
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model.to(device).eval()

# Dataset
assert os.path.isdir(DATA_ROOT), f"DATA_ROOT non trovato: {DATA_ROOT}"
split_path = os.path.join(DATA_ROOT, SPLIT)
assert os.path.isdir(split_path), f"Split '{SPLIT}' non trovato in {DATA_ROOT}"
ds = ImageFolder(split_path, transform=preprocess)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
classnames = ds.classes
num_classes = len(classnames)
print("Classi:", num_classes)


In [ ]:

# === Prompts & temperature ===
def make_prompts(cname):
    return [t.format(cname) for t in TEMPLATES] + [f"{cname} architecture"]

all_prompts, class_offsets = [], [0]
for cname in classnames:
    ps = make_prompts(cname)
    all_prompts.extend(ps)
    class_offsets.append(len(all_prompts))

tokens = tokenizer(all_prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

TEMP_GRID = [0.5, 1.0, 2.0]


In [ ]:

# === Eval helpers ===
def evaluate_with_temp(temp):
    y_true, y_pred = [], []
    probs_collector = []
    model_dtype = next(model.parameters()).dtype

    with torch.no_grad():
        for imgs, labels in dl:
            imgs = imgs.to(device, dtype=model_dtype)
            img_features = model.encode_image(imgs)
            img_features = img_features / img_features.norm(dim=-1, keepdim=True)

            logit_scale = model.logit_scale.exp() * temp
            logits = logit_scale * (img_features @ text_features.T)

            class_logits = []
            for i in range(num_classes):
                s, e = class_offsets[i], class_offsets[i+1]
                class_logits.append(logits[:, s:e].mean(dim=1))
            class_logits = torch.stack(class_logits, dim=1)

            prob = class_logits.softmax(dim=-1)
            pred = prob.argmax(dim=-1)

            y_true.extend(labels.tolist())
            y_pred.extend(pred.tolist())
            probs_collector.append(prob.cpu().numpy())

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)
    probs = np.concatenate(probs_collector, axis=0)

    top1 = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    top5 = top_k_accuracy_score(y_true, probs, k=min(5, num_classes), labels=np.arange(num_classes))
    bal_acc = balanced_accuracy_score(y_true, y_pred)

    return top1, macro_f1, top5, bal_acc, y_true, y_pred, probs

best = None
for t in TEMP_GRID:
    top1, macro_f1, top5, bal_acc, *_ = evaluate_with_temp(t)
    print(f"temp={t:.2f} -> top1={top1:.4f}  macroF1={macro_f1:.4f}  top5={top5:.4f}  balAcc={bal_acc:.4f}")
    if best is None or top1 > best[0]:
        best = (top1, macro_f1, top5, bal_acc, t)

best_top1, best_macro_f1, best_top5, best_bal_acc, best_temp = best
print("\nBest temperature:", best_temp)


In [ ]:

# === Final eval + latency, report, confusion matrix ===
top1, macro_f1, top5, bal_acc, y_true, y_pred, probs = evaluate_with_temp(best_temp)

# Latency (ms/img)
def measure_latency(num_batches=20):
    dtype = next(model.parameters()).dtype
    # warmup
    it = iter(dl)
    for _ in range(3):
        try:
            imgs, _ = next(it)
        except StopIteration:
            it = iter(dl)
            imgs, _ = next(it)
        imgs = imgs.to(device, dtype=dtype)
        with torch.no_grad():
            _ = model.encode_image(imgs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    it = iter(dl)
    for _ in range(min(num_batches, max(1, len(ds)//dl.batch_size))):
        try:
            imgs, _ = next(it)
        except StopIteration:
            it = iter(dl)
            imgs, _ = next(it)
        imgs = imgs.to(device, dtype=dtype)
        t0 = time.time()
        with torch.no_grad():
            _ = model.encode_image(imgs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = time.time()-t0
        times.append((dt/imgs.size(0))*1000.0)
    return float(np.mean(times))

latency_ms = measure_latency()

# Report & Confusion Matrix
print("Accuracy:", f"{top1:.4f}")
from sklearn.metrics import classification_report
print(classification_report(y_true, y_pred, target_names=classnames, digits=4, zero_division=0))

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(cm, display_labels=classnames).plot(include_values=False, cmap="Blues", ax=ax, xticks_rotation=90)
ax.set_title("Confusion Matrix (counts)")
plt.tight_layout()
plt.show()

gpu_hours = 0.0
print(f"Top-1={top1:.4f} | Macro-F1={macro_f1:.4f} | Top-5={top5:.4f} | Balanced Acc={bal_acc:.4f}")
print(f"Latency: {latency_ms:.1f} ms/img | GPU-h: {gpu_hours:.2f}")


In [ ]:

# === Save CSV row ===
row = {
    "Timestamp": datetime.utcnow().isoformat(),
    "Model": MODEL_NAME,
    "Pretrained": PRETRAINED,
    "Scenario": f"Zero-shot ({SPLIT})",
    "Techniques": "Prompt ensemble + temperature search",
    "Top-1 Acc": round(float(top1), 4),
    "Macro-F1": round(float(macro_f1), 4),
    "Top-5 Acc": round(float(best_top5), 4),
    "Balanced Acc": round(float(bal_acc), 4),
    "GPU-h": 0.0,
    "Latency (ms/img)": round(float(latency_ms), 1),
    "Effort umano (h)": float(EFFORT_HOURS),
    "Note": NOTES,
    "Classes": int(len(classnames)),
    "Batch size": int(BATCH_SIZE)
}
df = pd.DataFrame([row])
display(df)

csv_local = "/content/benchmark_results.csv"
if os.path.exists(csv_local):
    old = pd.read_csv(csv_local)
    out = pd.concat([old, df], ignore_index=True)
else:
    out = df.copy()
out.to_csv(csv_local, index=False)

csv_drive = os.path.join(DATA_ROOT, "benchmark_results.csv")
try:
    if os.path.exists(csv_drive):
        oldd = pd.read_csv(csv_drive)
        outd = pd.concat([oldd, df], ignore_index=True)
    else:
        outd = df.copy()
    outd.to_csv(csv_drive, index=False)
    print("Saved CSV to:", csv_local, "and", csv_drive)
except Exception as e:
    print("Saved CSV to:", csv_local)
    print("Drive save failed:", e)
